# Corpus-scale interference run (140k images)

Runs the confirmatory experiment pre-registered in `docs/07-preregistration-2.md`.

**Before running:** Settings -> Accelerator -> **GPU T4 x2** (or P100), and
Settings -> Internet -> **On** (needed to clone the repo and stream the corpus).

**Session limits.** Kaggle kills the session at 12h. This run is chunked and
resumable: each chunk writes a shard, and re-running skips shards already on
disk. To survive across sessions, save `/kaggle/working/results/large` as a
Kaggle Dataset at the end of a session and attach it to the next one (see the
last cell). Do **not** treat resumability as licence to peek at interim
results and stop early - the pre-registration fixes the corpus in advance.

In [ ]:
!nvidia-smi
import torch, os
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('cpus', os.cpu_count())

In [ ]:
# Repo + dependencies. invisible-watermark needs onnxruntime explicitly -
# RivaGan's ONNX weights ship inside the pip package but silently fail to
# load without it.
!git clone -q https://github.com/meowlawat/deepfake-findings.git /kaggle/working/repo || (cd /kaggle/working/repo && git pull -q)
!pip install -q invisible-watermark onnxruntime datasets 2>&1 | tail -2
%cd /kaggle/working/repo

In [ ]:
# Sanity: the same tooling verification that gates the local pipeline.
!python scripts/00_verify_tooling.py 2>&1 | grep -vE 'Loading weights|Warning:'

In [ ]:
# If a previous session's shards were saved as a Kaggle Dataset and attached,
# copy them in so this session resumes rather than restarting.
import shutil, os, glob
os.makedirs('results/large', exist_ok=True)
for prior in glob.glob('/kaggle/input/*/large'):
    print('resuming from', prior)
    shutil.copytree(prior, 'results/large', dirs_exist_ok=True)
print('shards present:', len(glob.glob('results/large/*/chunk_*.json')))

In [ ]:
# Throughput probe BEFORE committing to the full corpus. If this comes in far
# slower than expected, stop and rethink the plan rather than discovering it
# at hour 11.
!python scripts/e1_large.py --splits test --limit 200 --chunk-size 100 \
    --out-dir results/probe 2>&1 | grep -vE 'Loading weights|Warning:'

In [ ]:
# Extrapolate from the probe's img/s (printed above) and decide BEFORE the
# long run, rather than discovering the arithmetic at hour 11.
observed_rate = float(input('img/s from the probe cell above: '))
for n, label in ((20000, 'one split'), (140000, 'full corpus')):
    hours = n / observed_rate / 3600
    verdict = 'fits one 12h session' if hours < 11 else f'needs {int(hours//11)+1} sessions'
    print(f'{label:12s} n={n:7d}  ->  {hours:6.2f} h  ({verdict})')


## The confirmatory run

Split order is deliberate: `validation` and `train` are the confirmatory
splits for H1/H2 (never scored), and `train` additionally supplies the H3
leakage diagnostic. `test` is exploratory for H1/H2 because a slice of it
generated the hypothesis - it is still scored, for completeness and for the
H3 comparison, but it cannot confirm H1.

In [ ]:
# Full corpus. Expect this to run for hours; it prints progress per chunk and
# writes a shard each time, so an interrupted session loses at most one chunk.
!python scripts/e1_large.py \
    --splits validation test train \
    --chunk-size 500 \
    --out-dir results/large 2>&1 | grep -vE 'Loading weights|Warning:'

In [ ]:
# Aggregate: E0 by split, the H3 leakage diagnostic, and E1 with bootstrap CIs.
!python scripts/aggregate_large.py --in-dir results/large \
    --out results/large_summary.json --n-boot 1000

In [ ]:
# Evaluate the pre-registered hypotheses mechanically, so the verdict is not a
# judgement call made after seeing the numbers.
import json
s = json.load(open('results/large_summary.json'))
FLOOR = 0.10   # |delta_mu_net| effect-size floor, docs/07
GATE  = 0.02   # delta_auc_net band, pre-registration #1

print('H1 (delta_mu_net non-zero AND |effect| >= 0.10) on CONFIRMATORY splits:')
for r in s['interference']:
    if r['split'] == 'test':
        continue
    m = r['delta_mu_net']
    excl = (m['lo'] > 0) or (m['hi'] < 0)
    big  = abs(m['point']) >= FLOOR
    print(f"  {r['split']:11s} {r['detector']:9s} {r['scheme']:11s} "
          f"{m['point']:+.4f} [{m['lo']:+.4f},{m['hi']:+.4f}]  "
          f"{'SUPPORTS' if (excl and big) else 'does not support'}"
          f"{'' if big else '  (below effect floor)'}")

print('\nH2 (delta_auc_net stays within +/-0.02) on CONFIRMATORY splits:')
for r in s['interference']:
    if r['split'] == 'test':
        continue
    a = r['delta_auc_net']
    print(f"  {r['split']:11s} {r['detector']:9s} {r['scheme']:11s} "
          f"{a['point']:+.4f} [{a['lo']:+.4f},{a['hi']:+.4f}]  "
          f"{'HOLDS' if abs(a['point']) <= GATE else 'FAILS - preregistration #1 was underpowered'}")

print('\nH3 (leakage: train - test baseline AUC):')
for r in s.get('leakage', []):
    print(f"  {r['detector']:9s} gap={r['gap']:+.4f} -> {r['reading']}")

In [ ]:
# Package results for download / for attaching to a follow-on session.
!cd /kaggle/working/repo && tar czf /kaggle/working/results_large.tar.gz results/large results/large_summary.json
!ls -lh /kaggle/working/results_large.tar.gz
print('Download this, or Save Version -> Output, then attach it to the next session to resume.')